# DAM405 Machine Learning Operations Practical 4
## Automated Training and Tuning with FLAML

**Module tutor:** Kamal Acharya

This practical uses FLAML, an open-source AutoML library, to search for a strong classification model within a fixed time budget. The AutoML result is compared with a simple logistic-regression baseline, inspected, and saved for later use.

## Learning Outcomes

By the end of this practical, I should be able to:

1. Use an open-source AutoML tool to train and tune a model.
2. Compare an AutoML model with a simple baseline model.
3. Inspect the best model, its hyperparameters, and its score.
4. Save the resulting model for later use.

## Background

Automated machine learning (AutoML) automates repetitive model-building decisions, including algorithm selection and hyperparameter tuning. It does not remove the need for human decisions: the practitioner still chooses the dataset, target, evaluation metric, time budget, validation strategy, and whether the result is suitable for the application.

## 1. Install and Import Dependencies

Run the following cell once for the active notebook kernel. The `%pip` syntax installs packages into the environment associated with this notebook.

In [4]:
%pip install -q "flaml[automl]" scikit-learn joblib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.0/353.0 kB 29.2 MB/s eta 0:00:00


In [5]:
import time
from pathlib import Path

import joblib
import pandas as pd
from flaml import AutoML
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TIME_BUDGET_SECONDS = 60

## 2. Load and Split the Dataset

The built-in breast-cancer dataset is a binary classification problem (569 samples, 30 features). An 80/20 train/test split is used. The test set is kept separate from model training and is used only for the final comparison.

**Deviations from the handout (intentional):**
- `stratify=y` keeps the benign/malignant class ratio the same in the training and test sets, which makes the comparison fairer on a small test set.
- `random_state` / `seed` are fixed for the baseline and FLAML so the experiment is as reproducible as possible. FLAML's search is time-bounded, so results can still vary slightly between machines and runs.

In [6]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Dataset shape: {X.shape}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Dataset shape: (569, 30)
Training samples: 455
Test samples: 114


## 3. Build a Baseline Model

A baseline provides a reference point for judging whether AutoML adds value. Logistic regression is a transparent, strong starting point for this dataset. Its training time is also recorded so the compute cost can be compared with AutoML.

In [7]:
start = time.perf_counter()
baseline = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_seconds = time.perf_counter() - start

baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
baseline_roc_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])

print(f"Baseline test accuracy: {baseline_accuracy:.4f}")
print(f"Baseline test ROC AUC : {baseline_roc_auc:.4f}")
print(f"Baseline training time: {baseline_seconds:.2f} s")

Baseline test accuracy: 0.9649
Baseline test ROC AUC : 0.9954
Baseline training time: 0.57 s


## 4. Run FLAML AutoML

FLAML searches across candidate classifiers (LightGBM, random forest, XGBoost, extra trees, SGD, L1 logistic regression, ...) and their hyperparameters. It optimises cross-validated accuracy on the training data only; the test set is never seen during the search. The search is limited to 60 seconds so that the computational cost is explicit.

In [8]:

automl = AutoML()
automl.fit(
    X_train,
    y_train,
    task="classification",
    metric="accuracy",
    time_budget=TIME_BUDGET_SECONDS,
    seed=RANDOM_STATE,
)

[flaml.automl.logger: 10-05 09:00:29] {2470} INFO - task = classification
[flaml.automl.logger: 10-05 09:00:29] {2481} INFO - Evaluation method: cv
[flaml.automl.logger: 10-05 09:00:29] {2584} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 10-05 09:00:29] {2701} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 10-05 09:00:29] {3006} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 10-05 09:00:29] {3141} INFO - Estimated sufficient time budget=1736s. Estimated necessary time budget=40s.
[flaml.automl.logger: 10-05 09:00:29] {3192} INFO -  at 0.2s,	estimator lgbm's best error=9.4505e-02,	best estimator lgbm's best error=9.4505e-02
[flaml.automl.logger: 10-05 09:00:29] {3006} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 10-05 09:00:29] {3192} INFO -  at 0.3s,	estimator lgbm's best error=9.4505e-02,	best estimator lgbm's best error=9.4505e-02
[flaml.auto

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

[flaml.automl.logger: 10-05 09:00:30] {3192} INFO -  at 0.5s,	estimator lgbm's best error=7.9121e-02,	best estimator lgbm's best error=7.9121e-02
[flaml.automl.logger: 10-05 09:00:30] {3006} INFO - iteration 3, current learner sgd


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
INFO:flaml.tune.searcher.blendsearch:No low-cost p

[flaml.automl.logger: 10-05 09:00:30] {3192} INFO -  at 0.5s,	estimator sgd's best error=3.7363e-01,	best estimator lgbm's best error=7.9121e-02
[flaml.automl.logger: 10-05 09:00:30] {3006} INFO - iteration 4, current learner lgbm
[flaml.automl.logger: 10-05 09:00:30] {3192} INFO -  at 0.6s,	estimator lgbm's best error=4.8352e-02,	best estimator lgbm's best error=4.8352e-02
[flaml.automl.logger: 10-05 09:00:30] {3006} INFO - iteration 5, current learner lgbm


...

[flaml.automl.logger: 10-05 09:01:24] {3192} INFO -  at 55.1s,	estimator rf's best error=4.3956e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:24] {3006} INFO - iteration 174, current learner xgboost
[flaml.automl.logger: 10-05 09:01:25] {3192} INFO -  at 55.7s,	estimator xgboost's best error=2.6374e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:25] {3006} INFO - iteration 175, current learner xgboost
[flaml.automl.logger: 10-05 09:01:25] {3192} INFO -  at 55.9s,	estimator xgboost's best error=2.6374e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:25] {3006} INFO - iteration 176, current learner rf
[flaml.automl.logger: 10-05 09:01:25] {3192} INFO -  at 56.3s,	estimator rf's best error=4.3956e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:25] {3006} INFO - iteration 177, current learner sgd
[flaml.automl.logger: 10-05 09:01:25] {3192} INFO -  at

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[flaml.automl.logger: 10-05 09:01:28] {3192} INFO -  at 58.9s,	estimator rf's best error=3.9560e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:28] {3006} INFO - iteration 186, current learner xgboost
[flaml.automl.logger: 10-05 09:01:29] {3192} INFO -  at 59.7s,	estimator xgboost's best error=2.4176e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:29] {3006} INFO - iteration 187, current learner lrl1


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 10-05 09:01:29] {3192} INFO -  at 59.9s,	estimator lrl1's best error=9.0110e-02,	best estimator lgbm's best error=2.1978e-02
[flaml.automl.logger: 10-05 09:01:29] {3454} INFO - retrain lgbm for 0.1s
[flaml.automl.logger: 10-05 09:01:29] {3457} INFO - retrained model: LGBMClassifier(colsample_bytree=np.float64(0.9778726268416658),
               learning_rate=np.float64(0.5138753648083025), max_bin=31,
               min_child_samples=11, n_estimators=118, n_jobs=-1, num_leaves=7,
               reg_alpha=np.float64(0.014925762215576498),
               reg_lambda=np.float64(0.4520553510696902), verbose=-1)
[flaml.automl.logger: 10-05 09:01:29] {2731} INFO - fit succeeded
[flaml.automl.logger: 10-05 09:01:29] {2732} INFO - Time taken to find the best model: 18.232616901397705


## 5. Inspect the Best Model

The best estimator and configuration describe what FLAML selected. The cross-validation score is calculated as `1 - best_loss` because FLAML stores a loss (here `1 - accuracy`). It is the best validation score observed during the search and is different from the final test-set score below.

In [9]:
print("Best estimator:", automl.best_estimator)
print("Best configuration:")
for name, value in automl.best_config.items():
    print(f"  {name}: {value}")
print(f"Best cross-validation score: {1 - automl.best_loss:.4f}")

Best estimator: lgbm
Best configuration:
  n_estimators: 118
  num_leaves: 7
  min_child_samples: 11
  learning_rate: 0.5138753648083025
  log_max_bin: 5
  colsample_bytree: 0.9778726268416658
  reg_alpha: 0.014925762215576498
  reg_lambda: 0.4520553510696902
Best cross-validation score: 0.9780


## 6. Evaluate and Compare on the Test Set

Both models are evaluated on the same held-out test set. The difference is positive when AutoML performs better than the baseline, and is also reported in percentage points.

In [10]:
automl_accuracy = accuracy_score(y_test, automl.predict(X_test))
accuracy_difference = automl_accuracy - baseline_accuracy
wrong_baseline = round((1 - baseline_accuracy) * len(y_test))
wrong_automl = round((1 - automl_accuracy) * len(y_test))

print(f"AutoML test accuracy  : {automl_accuracy:.4f}  ({wrong_automl} errors / {len(y_test)})")
print(f"Baseline test accuracy: {baseline_accuracy:.4f}  ({wrong_baseline} errors / {len(y_test)})")
print(f"AutoML - baseline     : {accuracy_difference:+.4f} ({accuracy_difference * 100:+.2f} percentage points)")

AutoML test accuracy  : 0.9561  (5 errors / 114)
Baseline test accuracy: 0.9649  (4 errors / 114)
AutoML - baseline     : -0.0088 (-0.88 percentage points)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 7. Save the Best Model

The complete FLAML object is saved so that the selected model can be loaded and used later without repeating the search. The file is reloaded once to confirm that it works.

In [11]:
model_path = Path("automl_model.joblib")
joblib.dump(automl, model_path)
print(f"Saved {model_path}")

reloaded = joblib.load(model_path)
reloaded_accuracy = accuracy_score(y_test, reloaded.predict(X_test))
print(f"Reloaded model test accuracy: {reloaded_accuracy:.4f}")

Saved automl_model.joblib


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Reloaded model test accuracy: 0.9561


## 8. Optional Experiment: Time Budget and Metric

To study the cost/quality trade-off, the search is repeated with larger time budgets (120 s and 300 s) and once with `metric="roc_auc"`. ROC AUC measures how well the model ranks positive-class probabilities rather than how many labels are correct. For every run the best estimator, cross-validation score, test accuracy, test ROC AUC, and elapsed time are recorded. The models from these runs are only used for comparison; the saved model above is the 60-second accuracy run.

In [12]:
def run_automl(time_budget, metric):
    model = AutoML()
    start = time.perf_counter()
    model.fit(

        X_train,
        y_train,
        task="classification",
        metric=metric,
        time_budget=time_budget,
        seed=RANDOM_STATE,
        verbose=0,
    )
    elapsed = time.perf_counter() - start
    return {
        "metric": metric,
        "time_budget_s": time_budget,
        "elapsed_s": round(elapsed, 1),
        "best_estimator": model.best_estimator,
        "best_cv_score": round(1 - model.best_loss, 4),
        "test_accuracy": round(accuracy_score(y_test, model.predict(X_test)), 4),
        "test_roc_auc": round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4),
    }


experiments = [
    {
        "metric": "baseline (LogReg)",
        "time_budget_s": None,
        "elapsed_s": round(baseline_seconds, 1),
        "best_estimator": "logistic_regression",
        "best_cv_score": None,
        "test_accuracy": round(baseline_accuracy, 4),
        "test_roc_auc": round(baseline_roc_auc, 4),
    },
    {
        "metric": "accuracy",
        "time_budget_s": TIME_BUDGET_SECONDS,
        "elapsed_s": None,
        "best_estimator": automl.best_estimator,
        "best_cv_score": round(1 - automl.best_loss, 4),
        "test_accuracy": round(automl_accuracy, 4),
        "test_roc_auc": round(roc_auc_score(y_test, automl.predict_proba(X_test)[:, 1]), 4),
    },
]
experiments.append(run_automl(120, "accuracy"))
experiments.append(run_automl(300, "accuracy"))
experiments.append(run_automl(60, "roc_auc"))

results = pd.DataFrame(experiments)
results

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

,metric,time_budget_s,elapsed_s,best_estimator,best_cv_score,test_accuracy,test_roc_auc
0,baseline (LogReg),NaN,0.6,logistic_regression,NaN,0.9649,0.9954
1,accuracy,60.0,NaN,lgbm,0.9780,0.9561,0.9947
2,accuracy,120.0,120.1,xgboost,0.9802,0.9649,0.9944
3,accuracy,300.0,300.1,xgboost,0.9824,0.9474,0.9894
4,roc_auc,60.0,61.2,xgboost,0.9977,0.9649,0.9917


Note: the elapsed time of the main 60-second run is not re-measured here (it is shown in the FLAML log above and is approximately equal to the budget). FLAML stops once the budget is used, so elapsed time is close to `time_budget_s` for every AutoML run.